In [91]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import tensorflow as tf
from tensorflow import keras

In [30]:

movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

In [31]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [32]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [33]:
print(movies.shape)

(9742, 3)


In [34]:
print(ratings.shape)

(100836, 4)


In [35]:
print("Number of users:", ratings["userId"].nunique())
print("Number of movies rated:",ratings['movieId'].nunique())

Number of users: 610
Number of movies rated: 9724


In [36]:
all_genres = movies["genres"].str.split("|").explode().unique()

print(all_genres)
print("Number of genres :", len(all_genres))

<StringArray>
[         'Adventure',          'Animation',           'Children',
             'Comedy',            'Fantasy',            'Romance',
              'Drama',             'Action',              'Crime',
           'Thriller',             'Horror',            'Mystery',
             'Sci-Fi',                'War',            'Musical',
        'Documentary',               'IMAX',            'Western',
          'Film-Noir', '(no genres listed)']
Length: 20, dtype: str
Number of genres : 20


In [37]:
ratings['rating'].value_counts().sort_index()

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [38]:
ratings_per_user = ratings.groupby("userId")["rating"].count()
ratings_per_user.describe()

count     610.000000
mean      165.304918
std       269.480584
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
Name: rating, dtype: float64

In [39]:
ratings_per_movie = ratings.groupby('movieId')["rating"].count()
ratings_per_movie.describe()

count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
Name: rating, dtype: float64

In [40]:
all_genres = movies["genres"].str.split("|").explode().unique()
all_genres

<StringArray>
[         'Adventure',          'Animation',           'Children',
             'Comedy',            'Fantasy',            'Romance',
              'Drama',             'Action',              'Crime',
           'Thriller',             'Horror',            'Mystery',
             'Sci-Fi',                'War',            'Musical',
        'Documentary',               'IMAX',            'Western',
          'Film-Noir', '(no genres listed)']
Length: 20, dtype: str

In [41]:
for genre in all_genres:
  movies[genre] = movies["genres"].str.contains(genre).astype(int)
movies.head()

C:\Users\anshi\AppData\Local\Temp\ipykernel_28528\2052430026.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  movies[genre] = movies["genres"].str.contains(genre).astype(int)


,movieId,title,genres,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [42]:
avg_movie_rating = ratings.groupby("movieId")["rating"].mean()
movies["avg_rating"] = movies["movieId"].map(avg_movie_rating)
movies[["title", "avg_rating"]].head()

,title,avg_rating
0,Toy Story (1995),3.920930
1,Jumanji (1995),3.431818
2,Grumpier Old Men (1995),3.259615
3,Waiting to Exhale (1995),2.357143
4,Father of the Bride Part II (1995),3.071429


In [43]:
movies["year"] = movies['title'].str.extract(r"\((\d{4})\)").astype(float)

In [44]:
movie_features = movies[["movieId", "year", "avg_rating"] + list(all_genres)].copy()
# Impute NaN values in 'year' and 'avg_rating'
movie_features["year"] = movie_features["year"].fillna(movie_features["year"].median())
movie_features["avg_rating"] = movie_features["avg_rating"].fillna(movie_features["avg_rating"].mean())
movie_features = movie_features.set_index("movieId")
movie_features.head()

,year,avg_rating,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
movieId,,,,,,,,,,,,,,,,,,,,,
1,1995.0,3.920930,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1995.0,3.431818,1,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1995.0,3.259615,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1995.0,2.357143,0,0,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
5,1995.0,3.071429,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:
movies.columns

Index(['movieId', 'title', 'genres', 'Adventure', 'Animation', 'Children',
       'Comedy', 'Fantasy', 'Romance', 'Drama', 'Action', 'Crime', 'Thriller',
       'Horror', 'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir', '(no genres listed)', 'avg_rating', 'year'],
      dtype='str')

In [46]:
movies[["title", "year"]].head()



,title,year
0,Toy Story (1995),1995.0
1,Jumanji (1995),1995.0
2,Grumpier Old Men (1995),1995.0
3,Waiting to Exhale (1995),1995.0
4,Father of the Bride Part II (1995),1995.0


In [47]:
avg_movie_rating = ratings.groupby("movieId")["rating"].mean()
movies["avg_rating"] = movies["movieId"].map(avg_movie_rating)
movies[["title", "avg_rating"]].head()

,title,avg_rating
0,Toy Story (1995),3.920930
1,Jumanji (1995),3.431818
2,Grumpier Old Men (1995),3.259615
3,Waiting to Exhale (1995),2.357143
4,Father of the Bride Part II (1995),3.071429


In [48]:
movie_features = movies[["movieId", "year", "avg_rating"] + list(all_genres)].copy()
# Impute NaN values in 'year' and 'avg_rating'
movie_features["year"] = movie_features["year"].fillna(movie_features["year"].median())
movie_features["avg_rating"] = movie_features["avg_rating"].fillna(movie_features["avg_rating"].mean())
movie_features = movie_features.set_index("movieId")
movie_features.head()

,year,avg_rating,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
movieId,,,,,,,,,,,,,,,,,,,,,
1,1995.0,3.920930,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1995.0,3.431818,1,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1995.0,3.259615,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1995.0,2.357143,0,0,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
5,1995.0,3.071429,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [49]:
ratings_with_genre = ratings.merge(
    movies[["movieId"] + list(all_genres)],
    how="left"
)
ratings_with_genre.head()

,userId,movieId,rating,timestamp,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,1,4.0,964982703,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1,3,4.0,964981247,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,6,4.0,964982224,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,47,5.0,964983815,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,1,50,5.0,964982931,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [50]:
for genre in all_genres:
  ratings_with_genre[genre] = ratings_with_genre[genre] * ratings_with_genre["rating"]
ratings_with_genre.head()

,userId,movieId,rating,timestamp,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,1,4.0,964982703,4.0,4.0,4.0,4.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,3,4.0,964981247,0.0,0.0,0.0,4.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,6,4.0,964982224,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,47,5.0,964983815,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,50,5.0,964982931,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [51]:
user_genre_preference = ratings_with_genre.groupby("userId")[list(all_genres)].mean()

In [52]:
user_genre_preference.head()

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
userId,,,,,,,,,,,,,,,,,,,,
1,1.607759,0.586207,0.823276,1.530172,0.870690,0.482759,1.327586,1.676724,0.844828,0.982759,0.254310,0.323276,0.728448,0.426724,0.443966,0.000000,0.000000,0.129310,0.021552,0.0
2,0.431034,0.000000,0.000000,0.965517,0.000000,0.155172,2.275862,1.500000,1.310345,1.275862,0.103448,0.275862,0.534483,0.155172,0.000000,0.448276,0.517241,0.120690,0.000000,0.0
3,0.769231,0.051282,0.064103,0.230769,0.346154,0.064103,0.307692,1.282051,0.025641,0.743590,0.961538,0.128205,1.615385,0.064103,0.012821,0.000000,0.000000,0.000000,0.000000,0.0
4,0.490741,0.111111,0.175926,1.689815,0.324074,0.907407,1.935185,0.384259,0.476852,0.625000,0.078704,0.370370,0.157407,0.115741,0.296296,0.037037,0.013889,0.175926,0.074074,0.0
5,0.590909,0.590909,0.840909,1.181818,0.659091,0.772727,2.159091,0.636364,1.045455,0.727273,0.068182,0.090909,0.113636,0.227273,0.500000,0.000000,0.250000,0.136364,0.000000,0.0


In [53]:
user_avg_rating = ratings.groupby("userId")["rating"].mean()
user_genre_preference["avg_rating"] = user_avg_rating
user_genre_preference.head()

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,...,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed),avg_rating
userId,,,,,,,,,,,,,,,,,,,,,
1,1.607759,0.586207,0.823276,1.530172,0.870690,0.482759,1.327586,1.676724,0.844828,0.982759,...,0.323276,0.728448,0.426724,0.443966,0.000000,0.000000,0.129310,0.021552,0.0,4.366379
2,0.431034,0.000000,0.000000,0.965517,0.000000,0.155172,2.275862,1.500000,1.310345,1.275862,...,0.275862,0.534483,0.155172,0.000000,0.448276,0.517241,0.120690,0.000000,0.0,3.948276
3,0.769231,0.051282,0.064103,0.230769,0.346154,0.064103,0.307692,1.282051,0.025641,0.743590,...,0.128205,1.615385,0.064103,0.012821,0.000000,0.000000,0.000000,0.000000,0.0,2.435897
4,0.490741,0.111111,0.175926,1.689815,0.324074,0.907407,1.935185,0.384259,0.476852,0.625000,...,0.370370,0.157407,0.115741,0.296296,0.037037,0.013889,0.175926,0.074074,0.0,3.555556
5,0.590909,0.590909,0.840909,1.181818,0.659091,0.772727,2.159091,0.636364,1.045455,0.727273,...,0.090909,0.113636,0.227273,0.500000,0.000000,0.250000,0.136364,0.000000,0.0,3.636364


In [54]:
ratings_with_users = ratings.merge(
    user_genre_preference,
    on="userId",
    how="left"
)
ratings_with_users.head()

,userId,movieId,rating,timestamp,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed),avg_rating
0,1,1,4.0,964982703,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0.323276,0.728448,0.426724,0.443966,0.0,0.0,0.12931,0.021552,0.0,4.366379
1,1,3,4.0,964981247,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0.323276,0.728448,0.426724,0.443966,0.0,0.0,0.12931,0.021552,0.0,4.366379
2,1,6,4.0,964982224,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0.323276,0.728448,0.426724,0.443966,0.0,0.0,0.12931,0.021552,0.0,4.366379
3,1,47,5.0,964983815,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0.323276,0.728448,0.426724,0.443966,0.0,0.0,0.12931,0.021552,0.0,4.366379
4,1,50,5.0,964982931,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0.323276,0.728448,0.426724,0.443966,0.0,0.0,0.12931,0.021552,0.0,4.366379


In [55]:
ratings_full = ratings_with_users.merge(
    movie_features,
    left_on="movieId",
    right_index=True,
    how="left",
    suffixes=('_user', '_movie') # Added suffixes here
)
ratings_full.head()

,userId,movieId,rating,timestamp,Adventure_user,Animation_user,Children_user,Comedy_user,Fantasy_user,Romance_user,...,Horror_movie,Mystery_movie,Sci-Fi_movie,War_movie,Musical_movie,Documentary_movie,IMAX_movie,Western_movie,Film-Noir_movie,(no genres listed)_movie
0,1,1,4.0,964982703,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0,0,0,0,0,0,0,0,0,0
1,1,3,4.0,964981247,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0,0,0,0,0,0,0,0,0,0
2,1,6,4.0,964982224,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0,0,0,0,0,0,0,0,0,0
3,1,47,5.0,964983815,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0,1,0,0,0,0,0,0,0,0
4,1,50,5.0,964982931,1.607759,0.586207,0.823276,1.530172,0.87069,0.482759,...,0,1,0,0,0,0,0,0,0,0


In [56]:
y = ratings_full["rating"]

# Define original column names for user and movie features
original_user_cols = user_genre_preference.columns.tolist()
original_movie_cols = movie_features.columns.tolist()

print("Columns in ratings_full:", ratings_full.columns.tolist())

# Select user features with '_user' suffix from ratings_full
user_feature_cols_in_full = [col + '_user' for col in original_user_cols]
X_user = ratings_full[user_feature_cols_in_full]
# Rename columns back to original names for consistency
X_user.columns = original_user_cols

# Select movie features: 'year' (no suffix) and other features with '_movie' suffix
# 'year' is unique to movie_features, so it doesn't get a suffix from the merge
movie_feature_cols_in_full = ['year'] + [col + '_movie' for col in original_movie_cols if col != 'year']
X_movie = ratings_full[movie_feature_cols_in_full]
# Rename columns back to original names for consistency
X_movie.columns = original_movie_cols

print("NaNs in X_user before scaling:\n", X_user.isnull().sum())
print("NaNs in X_movie before scaling:\n", X_movie.isnull().sum())

print(X_user.shape, X_movie.shape, y.shape)

Columns in ratings_full: ['userId', 'movieId', 'rating', 'timestamp', 'Adventure_user', 'Animation_user', 'Children_user', 'Comedy_user', 'Fantasy_user', 'Romance_user', 'Drama_user', 'Action_user', 'Crime_user', 'Thriller_user', 'Horror_user', 'Mystery_user', 'Sci-Fi_user', 'War_user', 'Musical_user', 'Documentary_user', 'IMAX_user', 'Western_user', 'Film-Noir_user', '(no genres listed)_user', 'avg_rating_user', 'year', 'avg_rating_movie', 'Adventure_movie', 'Animation_movie', 'Children_movie', 'Comedy_movie', 'Fantasy_movie', 'Romance_movie', 'Drama_movie', 'Action_movie', 'Crime_movie', 'Thriller_movie', 'Horror_movie', 'Mystery_movie', 'Sci-Fi_movie', 'War_movie', 'Musical_movie', 'Documentary_movie', 'IMAX_movie', 'Western_movie', 'Film-Noir_movie', '(no genres listed)_movie']
NaNs in X_user before scaling:
 Adventure             0
Animation             0
Children              0
Comedy                0
Fantasy               0
Romance               0
Drama                 0
Action 

In [57]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [58]:
scaler_user = StandardScaler()
scaler_movie = StandardScaler()

X_user_scaled = scaler_user.fit_transform(X_user)
X_movie_scaled = scaler_movie.fit_transform(X_movie)

In [59]:
scaler_target = MinMaxScaler((0, 1))
y_scaled = scaler_target.fit_transform(y.values.reshape(-1, 1))

In [60]:
print(X_user_scaled.mean(), X_movie_scaled.mean())
print(y_scaled.min(), y_scaled.max())

1.7609585188155617e-17 -3.0809941091671605e-16
0.0 1.0


In [62]:
print("User feature shape:", X_user.shape)
print("Movie fetaure shape", X_movie.shape)
print("Target shape:", y.shape)

User feature shape: (100836, 21)
Movie fetaure shape (100836, 22)
Target shape: (100836,)


In [63]:
scaler_user = StandardScaler()
X_user_scaled = scaler_user.fit_transform(X_user)
scaler_movie = StandardScaler()
X_movie_scaled = scaler_movie.fit_transform(X_movie)
scaler_target = MinMaxScaler((0, 1))
y_scaled = scaler_target.fit_transform(y.values.reshape(-1, 1))

In [64]:
X_user_train, X_user_test, \
X_movie_train, X_movie_test, \
y_train, y_test = train_test_split(
    X_user_scaled,
    X_movie_scaled,
    y_scaled,
    train_size = 0.8,
    random_state = 1
)

In [65]:
print("User train shape:", X_user_train.shape)
print("Movie train shape:", X_movie_train.shape)
print("y train shape:", y_train.shape)

print("User test shape:", X_user_test.shape)
print("Movie test shape:", X_movie_test.shape)
print("y test shape:", y_test.shape)

User train shape: (80668, 21)
Movie train shape: (80668, 22)
y train shape: (80668, 1)
User test shape: (20168, 21)
Movie test shape: (20168, 22)
y test shape: (20168, 1)


In [66]:
embedding_size = 32
user_nn = keras.Sequential([
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(embedding_size)
])

movie_nn= keras.Sequential([
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(embedding_size)
])

user_input = keras.Input(shape=(X_user_train.shape[1],))
movie_input = keras.Input(shape=(X_movie_train.shape[1],))
user_vec = keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(user_nn(user_input))
movie_vec = keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(movie_nn(movie_input))
output = keras.layers.Dot(axes=1) ([user_vec, movie_vec])
model = keras.Model([user_input, movie_input], output)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 21)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 22)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32)        │     13,152 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32)        │     13,280 │ input_layer_1[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 32)        │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 32)        │          0 │ sequential_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ lambda[0][0],     │
│                     │                   │            │ lambda_1[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,432 (103.25 KB)

 Trainable params: 26,432 (103.25 KB)

 Non-trainable params: 0 (0.00 B)

In [67]:
tf.random.set_seed(1)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.MeanSquaredError()
)
history = model.fit(
    [X_user_train, X_movie_train],
    y_train,
    validation_data=([X_user_test, X_movie_test], y_test),
    epochs=30,
    batch_size=256
)

Epoch 1/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 0.0355 - val_loss: 0.0332
Epoch 2/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0320 - val_loss: 0.0318
Epoch 3/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0316 - val_loss: 0.0313
Epoch 4/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0312 - val_loss: 0.0311
Epoch 5/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0309 - val_loss: 0.0310
Epoch 6/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0307 - val_loss: 0.0308
Epoch 7/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0304 - val_loss: 0.0307
Epoch 8/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0302 - val_loss: 0.0306
Epoch 9/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0301 - val_loss: 0.0305
Epoch 10/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0300 - val_loss: 0.0304
Epoch 11/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0298 - val_loss: 0.0304
Epoch 12/30
316/316 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

In [68]:
model.evaluate([X_user_test, X_movie_test], y_test)

631/631 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0300


0.03003523498773575

In [69]:
X_user.columns


Index(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy', 'Romance',
       'Drama', 'Action', 'Crime', 'Thriller', 'Horror', 'Mystery', 'Sci-Fi',
       'War', 'Musical', 'Documentary', 'IMAX', 'Western', 'Film-Noir',
       '(no genres listed)', 'avg_rating'],
      dtype='str')

In [70]:
X_user.columns.duplicated()

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False])

In [71]:
X_user = X_user.loc[:, ~X_user.columns.duplicated()]

In [72]:
X_user.columns

Index(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy', 'Romance',
       'Drama', 'Action', 'Crime', 'Thriller', 'Horror', 'Mystery', 'Sci-Fi',
       'War', 'Musical', 'Documentary', 'IMAX', 'Western', 'Film-Noir',
       '(no genres listed)', 'avg_rating'],
      dtype='str')

In [73]:
new_user_dict ={
    'Adventure' : 0,
    'Animation' : 3,
    'Children' : 5,
    'Comedy' : 3,
    'Fantasy' : 0,
    'Romance' : 0,
    'Drama' : 5,
    'Action' : 0,
    'Crime': 0,
    'Thriller' : 0,
    'Horror' : 0,
    'Mystery' : 0,
    'Sci-Fi' : 0,
    'War' : 0,
    'Musical' : 0,
    'Documentary' : 0,
    'IMAX' : 0,
    'Western' : 0,
    'Film-Noir': 0,
    '(no genres listed)': 0,
}

ratings = [v for v in new_user_dict.values() if v > 0]
new_user_dict['avg_rating'] = np.mean(ratings) if ratings else 0
new_user_features = pd.DataFrame(
    [new_user_dict],
    columns=X_user.columns
)
new_user_features

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,...,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed),avg_rating
0,0,3,5,3,0,0,5,0,0,0,...,0,0,0,0,0,0,0,0,0,4.0


In [74]:
new_user_features.shape

(1, 21)

In [75]:
new_user_scaled = scaler_user.transform(new_user_features)
new_user_scaled.shape

(1, 21)

In [76]:
num_movies = X_movie_scaled.shape[0]
user_matrix = np.repeat(new_user_scaled, num_movies, axis=0)
user_matrix.shape


(100836, 21)

In [77]:
num_movies = X_movie_scaled.shape[0]
user_matrix = np.repeat(new_user_scaled, num_movies, axis=0)
print("User matrix shape:", user_matrix.shape)
print("Movie matrix shape:", X_movie_scaled.shape)

User matrix shape: (100836, 21)
Movie matrix shape: (100836, 22)


In [78]:
scores = model.predict([user_matrix, X_movie_scaled])
scores.shape

3152/3152 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step


(100836, 1)

In [79]:
scaled_unique_movie_features = scaler_movie.transform(movie_features)

num_unique_movies = movie_features.shape[0]
user_matrix_for_recommendation = np.repeat(new_user_scaled, num_unique_movies, axis=0)

scores = model.predict([user_matrix_for_recommendation, scaled_unique_movie_features])

predicted_ratings = scaler_target.inverse_transform(scores)

movie_features_with_predictions = movie_features.copy()
movie_features_with_predictions["predicted_rating"] = predicted_ratings

recommendations = movies.merge(
    movie_features_with_predictions[["predicted_rating"]],
    left_on="movieId",
    right_index=True,
    how="left"
)

top_recommendations = recommendations.sort_values(by="predicted_rating", ascending=False).reset_index(drop=True)

print("Top 10 Movie Recommendations for the New User:")
print(top_recommendations[["title", "predicted_rating"]].head(10))

305/305 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Top 10 Movie Recommendations for the New User:
                                              title  predicted_rating
0                                Winnie Pooh (1969)          4.874856
1                         Gena the Crocodile (1969)          4.874856
2                  Scooby-Doo Goes Hollywood (1979)          4.865407
3  On the Trail of the Bremen Town Musicians (1973)          4.860745
4                            Karlson Returns (1970)          4.860513
5                       There Once Was a Dog (1982)          4.859095
6                         Junior and Karlson (1968)          4.858129
7                     Dr. Phibes Rises Again (1972)          4.818310
8        In the blue sea, in the white foam. (1984)          4.816812
9                             Carry on Cabby (1963)          4.807111


**Reasoning**:
The first step is to inverse transform the `scores` obtained from the model prediction back to the original rating scale using `scaler_target`. This will give us the actual predicted ratings.



In [80]:
predicted_ratings = scaler_target.inverse_transform(scores)
print("Shape of predicted_ratings:", predicted_ratings.shape)
print("First 5 predicted ratings:", predicted_ratings[:5])

Shape of predicted_ratings: (9742, 1)
First 5 predicted ratings: [[4.469181 ]
 [4.1315804]
 [3.7905014]
 [3.345152 ]
 [3.764558 ]]


**Reasoning**:
Following the instructions, I will create a copy of the `movie_features` DataFrame, add the `predicted_ratings` as a new column, and then merge it with the `movies` DataFrame to include the movie titles, forming the basis for recommendations.



In [81]:
movie_features_with_predictions = movie_features.copy()
movie_features_with_predictions["predicted_rating"] = predicted_ratings

recommendations = movies.merge(
    movie_features_with_predictions[["predicted_rating"]],
    left_on="movieId",
    right_index=True,
    how="left"
)

print("Shape of movie_features_with_predictions:", movie_features_with_predictions.shape)
print("Head of movie_features_with_predictions:")
print(movie_features_with_predictions.head())
print("\nShape of recommendations:", recommendations.shape)
print("Head of recommendations:")
print(recommendations.head())

Shape of movie_features_with_predictions: (9742, 23)
Head of movie_features_with_predictions:
           year  avg_rating  Adventure  Animation  Children  Comedy  Fantasy  \
movieId                                                                        
1        1995.0    3.920930          1          1         1       1        1   
2        1995.0    3.431818          1          0         1       0        1   
3        1995.0    3.259615          0          0         0       1        0   
4        1995.0    2.357143          0          0         0       1        0   
5        1995.0    3.071429          0          0         0       1        0   

         Romance  Drama  Action  ...  Mystery  Sci-Fi  War  Musical  \
movieId                          ...                                  
1              0      0       0  ...        0       0    0        0   
2              0      0       0  ...        0       0    0        0   
3              1      0       0  ...        0       0    0   

In [82]:
scaled_unique_movie_features = scaler_movie.transform(movie_features)

num_unique_movies = movie_features.shape[0]
user_matrix_for_recommendation = np.repeat(new_user_scaled, num_unique_movies, axis=0)

print("Shape of user_matrix_for_recommendation:", user_matrix_for_recommendation.shape)
print("Shape of scaled_unique_movie_features:", scaled_unique_movie_features.shape)

scores = model.predict([user_matrix_for_recommendation, scaled_unique_movie_features])
print("Shape of scores:", scores.shape)

Shape of user_matrix_for_recommendation: (9742, 21)
Shape of scaled_unique_movie_features: (9742, 22)
305/305 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Shape of scores: (9742, 1)


In [83]:
predicted_ratings = scaler_target.inverse_transform(scores)
print("Shape of predicted_ratings:", predicted_ratings.shape)
print("First 5 predicted ratings:", predicted_ratings[:5])

Shape of predicted_ratings: (9742, 1)
First 5 predicted ratings: [[4.469181 ]
 [4.1315804]
 [3.7905014]
 [3.345152 ]
 [3.764558 ]]


In [84]:
movie_features_with_predictions = movie_features.copy()
movie_features_with_predictions["predicted_rating"] = predicted_ratings

recommendations = movies.merge(
    movie_features_with_predictions[["predicted_rating"]],
    left_on="movieId",
    right_index=True,
    how="left"
)

print("Shape of movie_features_with_predictions:", movie_features_with_predictions.shape)
print("Head of movie_features_with_predictions:")
print(movie_features_with_predictions.head())
print("\nShape of recommendations:", recommendations.shape)
print("Head of recommendations:")
print(recommendations.head())

Shape of movie_features_with_predictions: (9742, 23)
Head of movie_features_with_predictions:
           year  avg_rating  Adventure  Animation  Children  Comedy  Fantasy  \
movieId                                                                        
1        1995.0    3.920930          1          1         1       1        1   
2        1995.0    3.431818          1          0         1       0        1   
3        1995.0    3.259615          0          0         0       1        0   
4        1995.0    2.357143          0          0         0       1        0   
5        1995.0    3.071429          0          0         0       1        0   

         Romance  Drama  Action  ...  Mystery  Sci-Fi  War  Musical  \
movieId                          ...                                  
1              0      0       0  ...        0       0    0        0   
2              0      0       0  ...        0       0    0        0   
3              1      0       0  ...        0       0    0   

In [85]:
scaled_unique_movie_features = scaler_movie.transform(movie_features)

num_unique_movies = movie_features.shape[0]
user_matrix_for_recommendation = np.repeat(new_user_scaled, num_unique_movies, axis=0)

scores = model.predict([user_matrix_for_recommendation, scaled_unique_movie_features])

predicted_ratings = scaler_target.inverse_transform(scores)

movie_features_with_predictions = movie_features.copy()
movie_features_with_predictions["predicted_rating"] = predicted_ratings

recommendations = movies.merge(
    movie_features_with_predictions[["predicted_rating"]],
    left_on="movieId",
    right_index=True,
    how="left"
)

top_recommendations = recommendations.sort_values(by="predicted_rating", ascending=False).reset_index(drop=True)

print("Top 10 Movie Recommendations for the New User:")
print(top_recommendations[["title", "predicted_rating"]].head(10))

305/305 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Top 10 Movie Recommendations for the New User:
                                              title  predicted_rating
0                                Winnie Pooh (1969)          4.874856
1                         Gena the Crocodile (1969)          4.874856
2                  Scooby-Doo Goes Hollywood (1979)          4.865407
3  On the Trail of the Bremen Town Musicians (1973)          4.860745
4                            Karlson Returns (1970)          4.860513
5                       There Once Was a Dog (1982)          4.859095
6                         Junior and Karlson (1968)          4.858129
7                     Dr. Phibes Rises Again (1972)          4.818310
8        In the blue sea, in the white foam. (1984)          4.816812
9                             Carry on Cabby (1963)          4.807111
